In [44]:
!pip install tavily
!pip install ddgs
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [45]:
from dotenv import load_dotenv
import os
from google.colab import userdata
from tavily import TavilyClient

# load environment variables from .env file
_ = load_dotenv()

os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')
# connect
client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

In [46]:
# run search
result = client.search("New about Oracle earnings?",
                       include_answer=True)

result

{'query': 'New about Oracle earnings?',
 'response_time': 1.43,
 'follow_up_questions': None,
 'answer': "Oracle's Q3 earnings exceeded expectations, with non-GAAP EPS of $2.26 and revenue of $16.1 billion, leading to a stock increase. The company raised its 2027 revenue outlook to $90 billion. Oracle's strong performance was driven by cloud computing and AI bookings.",
 'images': [],
 'results': [{'url': 'https://www.investing.com/equities/oracle-corp-earnings',
   'title': 'Oracle (ORCL) Earnings Date & Report - Investing.com',
   'content': "## Oracle (ORCL) Earnings Dates & Reports. * Oracle's Q2 2026 non-GAAP EPS reached $2.26, exceeding forecasts by 37.8%, while revenue grew 13% YoY to $16.1B, slightly below expectations; stock rose 0.67% after hours. Investing.com - Oracle (NYSE: ORCL) reported first quarter EPS of $1.47, $0.01 worse than the analyst estimate of $1.48. Investing.com -\xa0Shares in Oracle (NYSE:ORCL) surged in extended hours trading after the cloud-computing grou

In [47]:
# print the answer
result["answer"]

"Oracle's Q3 earnings exceeded expectations, with non-GAAP EPS of $2.26 and revenue of $16.1 billion, leading to a stock increase. The company raised its 2027 revenue outlook to $90 billion. Oracle's strong performance was driven by cloud computing and AI bookings."

## REGULAR SEARCH

In [48]:
city = "Haldwani"

query = f"""
    what is the current weather in {city}?
    Should I travel there today?
    "weather.com"
"""

In [49]:
import requests
from bs4 import BeautifulSoup
from ddgs import DDGS
import re

ddg = DDGS()

def search(query, max_results=6):
    try:
        results = ddg.text(query, max_results=max_results)
        return [i["href"] for i in results]
    except Exception as e:
        print(f"returning previous results due to exception reaching ddg.")
        results = [ # cover case where DDG rate limits due to high deeplearning.ai volume
            "https://weather.com/weather/today/l/USCA0987:1:US",
            "https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8",
        ]
        return results


for i in search(query):
    print(i)

https://www.easeweather.com/asia/india/uttarakhand/naini-tal/haldwani/today
https://www.accuweather.com/en/in/haldwani-cum-kathgodam/191344/hourly-weather-forecast/191344
https://weather.com/en-IN/weather/tenday/l/Haldwani+Uttarakhand?canonicalCityId=b4dfe365467278b790a2710e4c46469fdfdbf8e98dba38103001b07ccaf61ce4
https://www.meteoblue.com/en/weather/week/haldwani_india_1270498
https://www.weather-forecast.com/locations/Haldwani-Cum-Kathgodam/forecasts/latest
https://www.weatherapi.com/weather/q/haldwani-1115769


In [50]:
def scrape_weather_info(url):
    """Scrape content from the given URL"""
    if not url:
        return "Weather information could not be found."

    # fetch data
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return "Failed to retrieve the webpage."

    # parse result
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup

In [51]:
# use DuckDuckGo to find websites and take the first result
url = search(query)[0]

# scrape first wesbsite
soup = scrape_weather_info(url)

print(f"Website: {url}\n\n")
print(str(soup.body)[:50000]) # limit long outputs

Website: https://weather.com/weather/tenday/l/0bf19da9d88dd96a715972b18aef1dbffb79f474915ac41537c79fb0725d3847


<body class="inter_1db00677-module__WhbPaa__className font-sans"><div hidden=""><!--$--><!--/$--></div><script>(self.__next_s=self.__next_s||[]).push(["/wx-next-web/static/vendor/@mparticle/web-sdk@2.47.1/dist/mparticle.min.js",{}])</script><script>(self.__next_s=self.__next_s||[]).push(["https://weather.com/api/v1/script/dprSdkScript.js",{"id":"dprsdk-script"}])</script><script>(self.__next_s=self.__next_s||[]).push([0,{"children":"\n\t\t\t\t\tif (window.top?.DprSdk) {\n\t\t\t\t\t\tasync function initDprSdk() {\n\t\t\t\t\t\t\ttry {\n\t\t\t\t\t\t\t\tawait window.top?.DprSdk.init({\n\t\t\t\t\t\t\t\t\tgetApplicationInfo: () => ({ id: 'weather.com', version: '2.0.0' }),\n\t\t\t\t\t\t\t\t\tgetUserRegime: () => 'usa',\n\t\t\t\t\t\t\t\t});\n\t\t\t\t\t\t\t} catch (_error) {\n\t\t\t\t\t\t\t\t// do nothing.\n\t\t\t\t\t\t\t}\n\t\t\t\t\t\t}\n\t\t\t\t\t\tinitDprSdk();\n\t\t\t\t\t}\n\t\t

In [52]:
# extract text
weather_data = []
for tag in soup.find_all(['h1', 'h2', 'h3', 'p']):
    text = tag.get_text(" ", strip=True)
    weather_data.append(text)

# combine all elements into a single string
weather_data = "\n".join(weather_data)

# remove all spaces from the combined text
weather_data = re.sub(r'\s+', ' ', weather_data)

print(f"Website: {url}\n\n")
print(weather_data)

Website: https://weather.com/weather/tenday/l/0bf19da9d88dd96a715972b18aef1dbffb79f474915ac41537c79fb0725d3847


Home Home Today Today Hourly Hourly 10 Day 10 Day Monthly Monthly Radar Radar Video Video Home Home Today Today Hourly Hourly 10 Day 10 Day Monthly Monthly Radar Radar Video Video Daily Forecast Location Haldwani, Uttarakhand, India · as of 5:30 AM UTC Trending Now Severe Weather Outbreak, With Threat Of Strong Tornadoes, Forecast Through Wednesday Dangerous Severe Thunderstorm Outbreak Unfolds As Tornadoes, Hail Hammer Multiple States Twister Picks Up Debris In Central Texas 0:33 Severe Weather Maps Tracker: Radar, Warnings, Storm Reports And More Severe Weather Outbreak, With Threat Of Strong Tornadoes, Forecast Through Wednesday Dangerous Severe Thunderstorm Outbreak Unfolds As Tornadoes, Hail Hammer Multiple States Twister Picks Up Debris In Central Texas 0:33 Severe Weather Maps Tracker: Radar, Warnings, Storm Reports And More Home & Garden Don’t Make These Mistakes Whe

## Agentic Search

In [53]:
# run search
result = client.search(query, max_results=1)


In [54]:
result['answer']

In [55]:
result

{'query': 'what is the current weather in Haldwani?\n    Should I travel there today?\n    weather.com',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'title': 'Weather in Haldwani',
   'url': 'https://www.weatherapi.com/',
   'content': "{'location': {'name': 'Haldwani', 'region': 'Uttarakhand', 'country': 'India', 'lat': 29.2167, 'lon': 79.5167, 'tz_id': 'Asia/Kolkata', 'localtime_epoch': 1773206840, 'localtime': '2026-03-11 10:57'}, 'current': {'last_updated_epoch': 1773206100, 'last_updated': '2026-03-11 10:45', 'temp_c': 32.7, 'temp_f': 90.8, 'is_day': 1, 'condition': {'text': 'Sunny', 'icon': '//cdn.weatherapi.com/weather/64x64/day/113.png', 'code': 1000}, 'wind_mph': 5.4, 'wind_kph': 8.6, 'wind_degree': 212, 'wind_dir': 'SSW', 'pressure_mb': 1012.0, 'pressure_in': 29.89, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 13, 'cloud': 3, 'feelslike_c': 30.5, 'feelslike_f': 86.9, 'windchill_c': 32.7, 'windchill_f': 90.8, 'heatindex_c': 30.5, 'heatindex_f

In [56]:
# print first result
data = result["results"][0]["content"]

print(data)

{'location': {'name': 'Haldwani', 'region': 'Uttarakhand', 'country': 'India', 'lat': 29.2167, 'lon': 79.5167, 'tz_id': 'Asia/Kolkata', 'localtime_epoch': 1773206840, 'localtime': '2026-03-11 10:57'}, 'current': {'last_updated_epoch': 1773206100, 'last_updated': '2026-03-11 10:45', 'temp_c': 32.7, 'temp_f': 90.8, 'is_day': 1, 'condition': {'text': 'Sunny', 'icon': '//cdn.weatherapi.com/weather/64x64/day/113.png', 'code': 1000}, 'wind_mph': 5.4, 'wind_kph': 8.6, 'wind_degree': 212, 'wind_dir': 'SSW', 'pressure_mb': 1012.0, 'pressure_in': 29.89, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 13, 'cloud': 3, 'feelslike_c': 30.5, 'feelslike_f': 86.9, 'windchill_c': 32.7, 'windchill_f': 90.8, 'heatindex_c': 30.5, 'heatindex_f': 86.9, 'dewpoint_c': 0.9, 'dewpoint_f': 33.5, 'vis_km': 10.0, 'vis_miles': 6.0, 'uv': 6.4, 'gust_mph': 6.2, 'gust_kph': 9.9}}


In [57]:
import json
from pygments import highlight, lexers, formatters

# parse JSON
parsed_json = json.loads(data.replace("'", '"'))

# pretty print JSON with syntax highlighting
formatted_json = json.dumps(parsed_json, indent=4)
colorful_json = highlight(formatted_json,
                          lexers.JsonLexer(),
                          formatters.TerminalFormatter())

print(colorful_json)


{
    "location": {
        "name": "Haldwani",
        "region": "Uttarakhand",
        "country": "India",
        "lat": 29.2167,
        "lon": 79.5167,
        "tz_id": "Asia/Kolkata",
        "localtime_epoch": 1773206840,
        "localtime": "2026-03-11 10:57"
    },
    "current": {
        "last_updated_epoch": 1773206100,
        "last_updated": "2026-03-11 10:45",
        "temp_c": 32.7,
        "temp_f": 90.8,
        "is_day": 1,
        "condition": {
            "text": "Sunny",
            "icon": "//cdn.weatherapi.com/weather/64x64/day/113.png",
            "code": 1000
        },
        "wind_mph": 5.4,
        "wind_kph": 8.6,
        "wind_degree": 212,
        "wind_dir": "SSW",
        "pressure_mb": 1012.0,
        "pressure_in": 29.89,
        "precip_mm": 0.0,
        "precip_in": 0.0,
        "humidity": 13,
        "cloud": 3,
        "feelslike_c": 30.5,
        "feelslike_f": 86.9,
        "windchill_c": 32.7,
        "windchill_f": 90.8,
        "heatind

## seach tool will return the data to LLM as json above as LLM only understand json response not the regular search response, LLM reasoning will deciede whether to travel to Haldwani or not.